<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/03-Catalogues-and-Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/mjcowley/Python_Intro/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics Research Group" width="400"/>

# Part 3: Catalogues and Data

**QUT Astrophysics Research Group (QARG) Python Training**

Group leads: Dr Michael Cowley and Dr Rossana Ruggeri
Author: Michael Cowley
Last updated: 18 Oct 2026

## Overview

So far you've typed every number in by hand. Real research starts with someone else's data: a catalogue from a survey, an image from a telescope, or the results of a query to an online archive. This part is about getting that data into Python, cleaning it up, and combining it.

Everything here uses **real data** from the CDFS field of the ZFOURGE survey ([Straatman et al. 2016](https://ui.adsabs.harvard.edu/abs/2016ApJ...830...51S)), which our group helped build and still uses. By the end, you'll reproduce a genuine galaxy evolution result: massive galaxies are much more likely to have stopped forming stars.

This notebook takes about **3 hours**.

### By the end of Part 3 you'll be able to

- Load tables into **pandas**, explore them, and select the rows and columns you need
- Deal with missing values and quality flags
- Read the three file formats you'll meet most: CSV, whitespace-delimited text and **FITS**
- Combine catalogues that share an ID column
- Open a FITS image and convert between pixel and sky coordinates
- Query an online archive with **astroquery**
- Cross-match two catalogues by position

## Setup

This cell imports everything we need and installs `astroquery` if it's missing (Colab doesn't include it by default).

It also defines `get_data()`, which downloads one of the course data files from GitHub and returns where it saved it. Files are only downloaded once, then reused. You can find details of every file, and where it came from, in the [data README](https://github.com/mjcowley/Python_Intro/tree/main/data).

In [ ]:
try:
    import astroquery
except ImportError:
    %pip install astroquery
    import astroquery

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.table import Table
from astropy.utils.data import download_file
from astropy.visualization import simple_norm
from astropy.wcs import WCS

DATA_URL = "https://raw.githubusercontent.com/mjcowley/Python_Intro/main/data/"


def get_data(filename):
    """Download a course data file (only the first time) and return its local path."""
    return download_file(DATA_URL + filename, cache=True)


print("pandas version:", pd.__version__)

---
## 1. Tables with pandas

A catalogue is a table: one row per object, one column per property. **pandas** is the standard Python library for tables. Its main object is the **DataFrame**.

### Look before you load

Before loading any file, look at the first few lines to see how it's laid out. Is there a header row? What separates the columns? How are missing values marked?

In [ ]:
path = get_data("zfourge_cdfs.csv")

with open(path) as f:           # 'with' closes the file for us when we're done
    for i in range(5):
        print(f.readline().strip())

It's a CSV (comma-separated values) file with a header row. Missing values look like they're marked `-99`. Let's load it:

In [ ]:
df = pd.read_csv(path)
df.head()

In a notebook, a DataFrame on the last line of a cell displays as a neat table. Here's what the columns mean:

| Column | Meaning |
|--------|---------|
| `id` | ZFOURGE ID number |
| `ra`, `dec` | Position in degrees |
| `f_Ks`, `e_Ks` | Flux and its uncertainty in the FourStar $K_s$ band ($2.2\,\mu$m) |
| `snr` | Signal-to-noise ratio in the detection image |
| `use` | 1 if the ZFOURGE team recommends the object for science, 0 otherwise |
| `star` | 1 if the object is likely a star |
| `nearstar` | 1 if the object is close to a bright star |
| `z_spec` | Spectroscopic redshift, if one has been measured |

Next, get an overview of the whole table:

In [ ]:
print(df.shape)         # (rows, columns)
print(df.columns.tolist())
df.info()

In [ ]:
df.describe()

`describe()` gives summary statistics for every column. Look at the `min` row: several columns have a minimum of exactly $-99$. That's the missing-value marker, and it's dragging every mean down. We'll fix this in Section 2.

### Selecting columns and rows

A single column is a **Series**. You select it with its name in square brackets, and it works with NumPy functions just like an array. Use a list of names to select several columns.

In [ ]:
print(df["snr"].median())

df[["id", "ra", "dec"]].head(3)

To select rows, use a boolean mask, exactly like with NumPy arrays in Part 2:

In [ ]:
bright = df[df["snr"] > 100]
print(f"{len(bright)} objects have S/N > 100")

stars = df[(df["star"] == 1) & (df["snr"] > 100)]
print(f"{len(stars)} of them are flagged as stars")

You can also select rows by position with `.iloc`, or by condition and column together with `.loc`:

In [ ]:
print(df.iloc[0])                                        # the first row, as a Series

df.loc[df["z_spec"] > 4, ["id", "z_spec", "snr"]]       # rows where z_spec > 4, three columns

### Sorting and counting

`sort_values()` sorts the table by a column, and `value_counts()` counts how often each value appears in a column.

In [ ]:
df.sort_values("snr", ascending=False).head(3)

In [ ]:
df["use"].value_counts()

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Exploring the Catalogue
  </h1>
</div>

Using `df`:

1. How many objects are flagged as stars? Store the answer in `n_stars`.
2. Find the 5 objects with the highest spectroscopic redshift, and show their `id`, `z_spec` and `snr`.
3. What's the median S/N of objects with `use == 1`, compared to objects with `use == 0`?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert n_stars == 337, "n_stars isn't right yet. Count the rows where star == 1."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
n_stars = (df["star"] == 1).sum()
print(f"{n_stars} stars")

print(df.sort_values("z_spec", ascending=False)[["id", "z_spec", "snr"]].head(5))

print("Median S/N, use == 1:", df.loc[df["use"] == 1, "snr"].median())
print("Median S/N, use == 0:", df.loc[df["use"] == 0, "snr"].median())
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 2. Missing Values and Quality Flags

### Missing values

The ZFOURGE team marked missing values with `-99`. pandas' marker for missing data is `NaN`, the same as NumPy's `np.nan`. Most pandas functions skip `NaN` automatically, which is what we want.

You can convert sentinel values while reading the file, using `na_values`:

In [ ]:
df = pd.read_csv(path, na_values=[-99])

print(df.isna().sum())       # number of missing values in each column

Most objects don't have a spectroscopic redshift, since spectra are expensive to get. Also, about a third of the objects have no FourStar $K_s$ flux. Why might that be?

### Quality flags

Most catalogues include **flags** that tell you which objects to trust. Here, `use = 1` marks objects that aren't stars, aren't close to bright stars, are detected with S/N ≥ 5 and have good coverage in the key filters. Objects outside the area covered by FourStar's $K_s$ imaging get `use = 0`, which explains most of the missing fluxes.

**Always read the catalogue paper or README before using the data.** Every survey defines its flags differently, and misunderstanding them is one of the easiest ways to get a wrong result.

In [ ]:
good = df[df["use"] == 1]

print(f"{len(good)} of {len(df)} objects have use = 1")
print("Missing Ks fluxes among them:", good["f_Ks"].isna().sum())

### Calculating magnitudes, and negative fluxes

ZFOURGE fluxes are on an AB zero point of 25. This means one flux unit is 0.3631 µJy, and the AB magnitude is

$$m_\mathrm{AB} = 25 - 2.5\log_{10}(f)$$

Watch what happens when we calculate magnitudes:

In [ ]:
ks_mag = 25 - 2.5 * np.log10(good["f_Ks"])

print("Objects with no magnitude:", ks_mag.isna().sum())
print("Objects with zero or negative flux:", (good["f_Ks"] <= 0).sum())

NumPy warns us that it took the log of zero or negative numbers. These are **real measurements**, not errors. A very faint object's measured flux scatters around its true value because of noise, and sometimes lands below zero. You can't turn a negative flux into a magnitude, but you shouldn't just throw these objects away either. If you averaged the fluxes of many faint objects (stacking, as in Part 2), the negative ones are needed to get the right answer.

When you only need magnitudes for plotting, it's fine to leave them as `NaN`. To avoid the warning, only take the log where the flux is positive, using pandas' `.where()`:

In [ ]:
f = good["f_Ks"]
ks_mag = 25 - 2.5 * np.log10(f.where(f > 0))      # .where() keeps values where the condition is True, NaN elsewhere
print(ks_mag.describe())

### Adding columns

Assigning to a new column name adds a column. When you're working on a selection of rows, use `.copy()` first so pandas knows you want a new, independent table:

In [ ]:
good = df[df["use"] == 1].copy()
good["Ks_mag"] = 25 - 2.5 * np.log10(good["f_Ks"].where(good["f_Ks"] > 0))

good[["id", "f_Ks", "Ks_mag"]].head()

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: A Clean Sample
  </h1>
</div>

1. Starting from `df`, make a new DataFrame called `clean` containing only objects with `use == 1` **and** a positive $K_s$ flux.
2. Add a column `Ks_mag` with the AB magnitude, and a column `Ks_snr` with the $K_s$ signal-to-noise ratio, `f_Ks / e_Ks`.
3. What's the median $K_s$ magnitude of the clean sample? What's the faintest magnitude at which objects have $K_s$ S/N > 5? *Hint: select the rows with `Ks_snr > 5` and find the maximum magnitude.*
4. How many objects in `clean` have a spectroscopic redshift?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert len(clean) == 12676, "clean should have 12,676 rows. Check both conditions."
assert "Ks_mag" in clean.columns and "Ks_snr" in clean.columns, "Add both new columns."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
clean = df[(df["use"] == 1) & (df["f_Ks"] > 0)].copy()

clean["Ks_mag"] = 25 - 2.5 * np.log10(clean["f_Ks"])
clean["Ks_snr"] = clean["f_Ks"] / clean["e_Ks"]

print(f"{len(clean)} objects")
print(f"Median Ks: {clean['Ks_mag'].median():.2f}")
print(f"Faintest with Ks S/N > 5: {clean.loc[clean['Ks_snr'] > 5, 'Ks_mag'].max():.2f}")
print(f"With a spectroscopic redshift: {clean['z_spec'].notna().sum()}")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 3. Other File Formats

### Whitespace-delimited text

Many older astronomy codes write plain text files with columns separated by spaces, and comment lines starting with `#`. The SED-fitting code FAST is one of them. Let's look at a FAST output (`.fout`) file, which has stellar masses and star formation rates for every ZFOURGE object:

In [ ]:
fast_path = get_data("zfourge_cdfs.fout")

with open(fast_path) as f:
    for i in range(6):
        print(f.readline().rstrip())

The column names are on a comment line, so pandas can't find them by itself. We tell it:

- `sep=r"\s+"`: columns are separated by one or more spaces
- `comment="#"`: ignore everything after a `#`
- `names=[...]`: the column names, copied from the header line

In [ ]:
fast_columns = ["id", "z", "ltau", "metal", "lage", "Av", "lmass", "lsfr", "lssfr", "la2t", "chi2"]

fast = pd.read_csv(fast_path, sep=r"\s+", comment="#", names=fast_columns, na_values=[-99])
fast.head()

The columns that start with `l` are base-10 logarithms: `lmass` is $\log_{10}(M_*/M_\odot)$, and `lsfr` is $\log_{10}$ of the star formation rate in $M_\odot$/yr. Astronomers often store quantities like these as logs because they span many orders of magnitude.

### FITS files

**FITS** (Flexible Image Transport System) is astronomy's standard file format. A FITS file holds one or more **HDUs** (Header Data Units). Each HDU has a **header** of metadata and some **data**, which can be an image or a table.

Astropy reads FITS files with `fits.open()`:

In [ ]:
eazy_path = get_data("zfourge_cdfs_eazy.fits")

hdul = fits.open(eazy_path)
hdul.info()

HDU 0 (the "primary" HDU) is empty, and the table is in HDU 1, which is named `EAZY`. This layout is very common. The header holds keywords describing the data:

In [ ]:
header = hdul[1].header
print(header["SURVEY"], header["FIELD"], header["REF"])
header

In [ ]:
hdul.close()      # close the file when you're done with it

For tables, it's easier to use Astropy's `Table`, which reads the table HDU directly. An Astropy Table is similar to a DataFrame and you'll see both in the literature. Convert between them with `.to_pandas()` and `Table.from_pandas()`.

In [ ]:
eazy_table = Table.read(eazy_path, hdu="EAZY")
print(len(eazy_table), "rows")
print(eazy_table.colnames)
eazy_table[:3]

This file holds the output of **EAZY**, the photometric redshift code used by ZFOURGE. `z_peak` is the photometric redshift, `l68` and `u68` are its 68% confidence limits, and `f_U`, `f_V` and `f_J` are the galaxy's fluxes in the rest-frame U, V and J filters, worked out from its fitted spectrum. These are what you need for a UVJ diagram.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Loading EAZY Results
  </h1>
</div>

1. Convert `eazy_table` to a pandas DataFrame called `eazy`, and replace the `-99` values with `NaN`. *Hint: DataFrames have a `.replace(old, new)` method.*
2. Check that `eazy` and `fast` have the same number of rows as `df`. Then check whether their `id` columns match `df["id"]` row by row. *Hint: `(a == b).all()` is `True` only if every element matches.* What do you find?
3. Calculate the typical photometric redshift uncertainty as half the width of the 68% interval, `(u68 - l68) / 2`, and print its median.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert isinstance(eazy, pd.DataFrame), "eazy should be a pandas DataFrame. Use .to_pandas()."
assert eazy["z_peak"].isna().sum() == 4, "Did you replace -99 with NaN?"
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
eazy = eazy_table.to_pandas().replace(-99, np.nan)

print(len(df), len(eazy), len(fast))
print((eazy["id"] == df["id"]).all(), (fast["id"] == df["id"]).all())     # False! The rows are in different orders
print(set(eazy["id"]) == set(df["id"]))                                    # but they hold the same objects

z_err = (eazy["u68"] - eazy["l68"]) / 2
print(f"Median photo-z uncertainty: {z_err.median():.3f}")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

The tables have the same number of rows, but the IDs don't line up! They contain the same objects, listed in a **different order**. This happens all the time with real data, for example when tables are sorted differently or downloaded from different places. If you'd put the columns side by side by row number, every galaxy would have been given another galaxy's redshift, and nothing would have warned you.

---
## 4. Combining Tables

We now have three tables about the same objects: photometry in `df`, redshifts and rest-frame fluxes in `eazy`, and stellar population results in `fast`. Combining them is called a **merge** (or a **join**). `pd.merge()` matches up rows that share the same value in a key column, here `id`.

As you found in the last exercise, these tables list their objects in different orders, so we can't just put them side by side. Even when rows do seem to line up, always merge on the ID. Relying on row order is a classic source of silent bugs.

In [ ]:
small_a = pd.DataFrame({"id": [1, 2, 3], "z": [0.5, 1.2, 2.1]})
small_b = pd.DataFrame({"id": [3, 1, 4], "lmass": [10.8, 9.9, 11.2]})

print(pd.merge(small_a, small_b, on="id"))               # inner: only ids in both tables
print()
print(pd.merge(small_a, small_b, on="id", how="left"))   # left: every row of small_a, NaN where no match

Notice that the rows of `small_b` were in a different order, but `merge` matched them up correctly anyway.

If both tables have a column with the same name (apart from the key), pandas adds the suffixes `_x` and `_y` to tell them apart. You can choose better names with the `suffixes` argument, or just select the columns you need before merging.

In [ ]:
eazy = eazy_table.to_pandas().replace(-99, np.nan)     # as in the exercise above, in case you skipped it

cat = pd.merge(df, eazy[["id", "z_peak", "l68", "u68", "f_U", "f_V", "f_J"]], on="id")
cat = pd.merge(cat, fast[["id", "z", "lmass", "lsfr", "Av"]].rename(columns={"z": "z_best"}), on="id")

print(cat.shape)
cat.head(3)

We renamed FAST's `z` column to `z_best` as we merged it. FAST fitted each galaxy at its spectroscopic redshift where there is one, and at its photometric redshift otherwise, so `z_best` is the redshift that matches the stellar masses. Use it whenever you select galaxies by redshift and then use their masses.

Here's why it matters:

In [ ]:
cat.loc[cat["id"] == 13914, ["id", "z_spec", "z_peak", "z_best", "lmass"]]

This galaxy's photometric redshift is 0.55, but its spectrum puts it at $z = 3.48$, a so-called **catastrophic outlier**. FAST used the spectroscopic redshift, so the mass of $10^{11.25}\,M_\odot$ is for a galaxy at $z = 3.48$. If you'd selected on `z_peak`, it would have landed in a low-redshift sample as an extremely massive galaxy. You'll look at how well photometric redshifts work in general in Part 5.

### Grouping

`groupby()` splits a table into groups and summarises each one. Combined with `pd.cut()`, which sorts values into bins, it's a quick way to see how one property changes with another. Here's the median stellar mass of good galaxies in bins of redshift:

In [ ]:
galaxies = cat[cat["use"] == 1].copy()
galaxies["z_bin"] = pd.cut(galaxies["z_best"], bins=[0, 0.5, 1, 1.5, 2, 3, 4])

galaxies.groupby("z_bin", observed=True)["lmass"].agg(["count", "median"])

At higher redshift we only see the more massive galaxies, because the less massive ones are too faint to detect. This is called **Malmquist bias**, and it affects almost every flux-limited survey. You'll need to think about it in your project.

### Saving your results

Save a DataFrame with `.to_csv()`, or convert it to an Astropy Table to write a FITS file:

In [ ]:
galaxies.drop(columns="z_bin").to_csv("zfourge_cdfs_galaxies.csv", index=False)

Table.from_pandas(galaxies.drop(columns="z_bin")).write("zfourge_cdfs_galaxies.fits", overwrite=True)

> **In Colab**, saved files disappear when your session ends. Download them from the *Files* panel on the left, or [mount your Google Drive](https://colab.research.google.com/notebooks/io.ipynb) and save there instead.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Quiescent Galaxies and Stellar Mass
  </h1>
</div>

Time for some real science. Rest-frame colours are calculated from flux ratios:

$$U-V = -2.5\log_{10}(f_U / f_V), \qquad V-J = -2.5\log_{10}(f_V / f_J)$$

and a galaxy is quiescent if $U-V > 1.3$, $V-J < 1.6$ and $U-V > 0.88\,(V-J) + 0.59$.

1. From `cat`, make a DataFrame called `sample` with `use == 1`, $0.5 < z_\mathrm{best} < 2.5$ and $\log_{10}(M_*/M_\odot) > 9$.
2. Add columns `u_v` and `v_j` with the rest-frame colours, and a boolean column `quiescent`.
3. Using `pd.cut()` with bins `[9, 9.5, 10, 10.5, 11, 12]` and `groupby()`, find the number of galaxies and the **fraction that are quiescent** in each mass bin. *Hint: the mean of a boolean column is the fraction that are `True`.*
4. How does the quiescent fraction change with stellar mass?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert len(sample) == 3698, "sample should have 3,698 galaxies. Check your three conditions."
assert sample["quiescent"].sum() == 422, "There should be 422 quiescent galaxies. Check the UVJ criteria."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
sample = cat[(cat["use"] == 1) & (cat["z_best"] > 0.5) & (cat["z_best"] < 2.5) & (cat["lmass"] > 9)].copy()

sample["u_v"] = -2.5 * np.log10(sample["f_U"] / sample["f_V"])
sample["v_j"] = -2.5 * np.log10(sample["f_V"] / sample["f_J"])
sample["quiescent"] = (sample["u_v"] > 1.3) & (sample["v_j"] < 1.6) & (sample["u_v"] > 0.88 * sample["v_j"] + 0.59)

sample["mass_bin"] = pd.cut(sample["lmass"], bins=[9, 9.5, 10, 10.5, 11, 12])
summary = sample.groupby("mass_bin", observed=True)["quiescent"].agg(["count", "mean"])
summary.columns = ["n_galaxies", "quiescent_fraction"]
print(summary)
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

The quiescent fraction climbs from a few percent at $10^9\,M_\odot$ to around half above $10^{11}\,M_\odot$. The most massive galaxies were the first to stop forming stars. Explaining why is one of the big open questions in galaxy evolution, and you've just measured it from a real survey.

You'll make the UVJ diagram for this sample in Part 4.

---
## 5. FITS Images and World Coordinates

FITS files also store images. Let's open an r-band image of part of CDFS from the DESI Legacy Imaging Surveys:

In [ ]:
image_path = get_data("legacy_cdfs_r.fits")

with fits.open(image_path) as hdul:
    hdul.info()
    image = hdul[0].data
    image_header = hdul[0].header

print(image.shape, image.dtype)
print("Units:", image_header["BUNIT"], "/", image_header.comments["BUNIT"])

The image is a 2D NumPy array, exactly like the simulated image in Part 2. Here's a quick look. `simple_norm` with an `asinh` stretch shows faint and bright features at the same time, which a linear scale can't do. Part 4 covers displaying images properly.

In [ ]:
norm = simple_norm(image, "asinh", percent=99.5)

plt.figure(figsize=(6, 6))
plt.imshow(image, origin="lower", cmap="gray_r", norm=norm)
plt.xlabel("x (pixels)")
plt.ylabel("y (pixels)")
plt.show()

### World Coordinate System (WCS)

How do we know which pixel corresponds to which point on the sky? The header contains a **World Coordinate System** (WCS), a description of how pixel positions map to RA and Dec. These are the `CTYPE`, `CRVAL`, `CRPIX` and `CD` keywords:

In [ ]:
for key in ["CTYPE1", "CTYPE2", "CRVAL1", "CRVAL2", "CRPIX1", "CRPIX2", "CD1_1", "CD2_2"]:
    print(f"{key:7s} = {image_header[key]}")

You never need to do the maths yourself. Astropy's `WCS` reads these keywords and does the conversion for you. Sky positions are represented as `SkyCoord` objects, which you'll use again in Section 7.

In [ ]:
wcs = WCS(image_header)

centre = wcs.pixel_to_world(349.5, 349.5)         # pixels count from 0 in Python
print(centre)
print(centre.to_string("hmsdms"))

In [ ]:
target = SkyCoord(ra=53.143833 * u.deg, dec=-27.813473 * u.deg)     # a bright galaxy at z = 0.21
x, y = wcs.world_to_pixel(target)
print(f"Pixel position: x = {x:.1f}, y = {y:.1f}")

### Cutouts

`Cutout2D` cuts a small piece out of an image around a sky position. It also gives you a new WCS for the cutout, so the coordinates stay correct.

In [ ]:
cutout = Cutout2D(image, position=target, size=20 * u.arcsec, wcs=wcs)
print(cutout.data.shape)

plt.figure(figsize=(4, 4))
plt.imshow(cutout.data, origin="lower", cmap="gray_r", norm=simple_norm(cutout.data, "asinh", percent=99.5))
plt.title("20 arcsec cutout")
plt.show()

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Finding Galaxies in the Image
  </h1>
</div>

The image covers only a small part of the ZFOURGE field.

1. Make a `SkyCoord` called `galaxy_coords` holding the positions of all the galaxies in `sample` (from the Section 4 exercise). *Hint: `SkyCoord(ra=sample["ra"].to_numpy() * u.deg, dec=...)`.*
2. Use `wcs.world_to_pixel()` to get their pixel positions, and make a mask called `in_image` that's `True` for galaxies that land inside the image (both pixel coordinates between 0 and 699).
3. How many galaxies from `sample` fall in the image? Of these, find the one with the highest stellar mass, and print its ID, redshift, mass and whether it's quiescent.
4. Make a 15 arcsec cutout around it and display it.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert in_image.sum() == 273, "There should be 273 galaxies in the image. Check your pixel limits."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
galaxy_coords = SkyCoord(ra=sample["ra"].to_numpy() * u.deg, dec=sample["dec"].to_numpy() * u.deg)

x, y = wcs.world_to_pixel(galaxy_coords)
in_image = (x >= 0) & (x <= 699) & (y >= 0) & (y <= 699)
print(f"{in_image.sum()} galaxies from the sample are in the image")

in_view = sample[in_image]
biggest = in_view.loc[in_view["lmass"].idxmax()]      # idxmax gives the row label of the maximum
print(f"Most massive: ID {biggest['id']}, z = {biggest['z_best']:.2f}, "
      f"log M = {biggest['lmass']:.2f}, quiescent: {biggest['quiescent']}")

position = SkyCoord(ra=biggest["ra"] * u.deg, dec=biggest["dec"] * u.deg)
cut = Cutout2D(image, position=position, size=15 * u.arcsec, wcs=wcs)
plt.imshow(cut.data, origin="lower", cmap="gray_r", norm=simple_norm(cut.data, "asinh", percent=99.5))
plt.title(f"ZFOURGE {biggest['id']}, z = {biggest['z_best']:.2f}")
plt.show()
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

The most massive galaxy in the image is barely visible! At $z = 1.55$, light we observe in the r band (about 620 nm) left the galaxy at about $620 / (1 + z) \approx 240$ nm, in the ultraviolet. A quiescent galaxy's old stars give out very little UV light, so it looks faint, even though it has more than $10^{11}$ solar masses of stars. This is exactly why ZFOURGE was designed around deep near-infrared imaging: at these redshifts, the near-infrared samples the rest-frame optical light from the bulk of a galaxy's stars.

---
## 6. Querying Online Archives

Most astronomical data lives in online archives, and **astroquery** lets you search them from Python. It has modules for most major archives, including VizieR, SIMBAD, NED, MAST (Hubble, JWST, TESS), IRSA (Spitzer, WISE), ESO and Gaia.

### VizieR

[VizieR](https://vizier.cds.unistra.fr/) hosts the tables from tens of thousands of published papers. Every catalogue has an ID, usually built from the journal, volume and page. The ZFOURGE catalogues are `J/ApJ/830/51`:

In [ ]:
from astroquery.vizier import Vizier

try:
    zfourge_tables = Vizier.get_catalogs("J/ApJ/830/51")
    print(zfourge_tables)
except Exception as e:
    print("Couldn't reach VizieR:", e)

That's every table from the ZFOURGE release, and the files you've been using came from four of them. By default VizieR only returns the first 50 rows of each table. Use `Vizier(row_limit=-1)` to get everything, and `columns=[...]` to choose which columns you want.

Archives are sometimes slow or down, so the `try` and `except` stops the notebook breaking if the query fails.

### Searching by position

The most common query is a **cone search**: everything within some radius of a position. Let's get the Gaia DR3 stars around the centre of CDFS. Gaia's catalogue on VizieR is `I/355/gaiadr3`.

In [ ]:
cdfs_centre = SkyCoord(ra=53.12 * u.deg, dec=-27.81 * u.deg)

gaia_query = Vizier(columns=["Source", "RA_ICRS", "DE_ICRS", "Gmag", "Plx", "e_Plx", "pmRA", "pmDE"], row_limit=-1)

try:
    result = gaia_query.query_region(cdfs_centre, radius=12 * u.arcmin, catalog="I/355/gaiadr3")
    print(f"Found {len(result[0])} Gaia sources")
    print(result[0][:5])
except Exception as e:
    print("Couldn't reach VizieR:", e)

To make sure everyone gets the same results, we saved the output of this query when the notebook was written. Archives update their data releases from time to time, so it's good practice to save a copy of the data you used in a project, and to record when you downloaded it.

In [ ]:
gaia = pd.read_csv(get_data("gaia_dr3_cdfs.csv"))
print(len(gaia), "Gaia sources")
gaia.head()

Gaia measures the **parallax** (in milliarcseconds) and **proper motion** (in mas/yr) of each source. A star in our galaxy is close enough to have a measurable parallax and motion, but a distant galaxy has neither. Gaia also misses most of the galaxies in ZFOURGE, since they're far too faint.

---
## 7. Cross-Matching Catalogues

Different surveys give their objects different IDs, so to combine them we have to match objects by **position**. This is called cross-matching. Astropy does it with `SkyCoord.match_to_catalog_sky()`, which finds the nearest object in another catalogue for every object in the first.

In [ ]:
zf_coords = SkyCoord(ra=df["ra"].to_numpy() * u.deg, dec=df["dec"].to_numpy() * u.deg)
gaia_coords = SkyCoord(ra=gaia["ra"].to_numpy() * u.deg, dec=gaia["dec"].to_numpy() * u.deg)

idx, sep, _ = gaia_coords.match_to_catalog_sky(zf_coords)

print(idx[:5])           # for each Gaia source, the row number of the nearest ZFOURGE object
print(sep[:5].arcsec)    # and how far away it is

Every Gaia source gets a "nearest" ZFOURGE object, even if it's far away, for example because it lies outside the ZFOURGE field. So we have to decide what counts as a real match. A histogram of the separations helps:

In [ ]:
plt.hist(sep.arcsec, bins=np.logspace(-2, 3, 50))
plt.xscale("log")
plt.axvline(1, color="k", linestyle="--")
plt.xlabel("Separation to nearest ZFOURGE object (arcsec)")
plt.ylabel("Number of Gaia sources")
plt.show()

There's a clear peak below 1 arcsec, where the true matches are, and a long tail of unrelated objects much further away. We'll take 1 arcsec as our matching radius.

In [ ]:
matched = sep < 1 * u.arcsec
print(f"{matched.sum()} of {len(gaia)} Gaia sources have a ZFOURGE counterpart within 1 arcsec")

# Put the matched pairs side by side in one table
pairs = gaia[matched].reset_index(drop=True)
zf_match = df.iloc[idx[matched]].reset_index(drop=True)
pairs = pairs.join(zf_match[["id", "star", "use", "f_Ks"]])
pairs["sep_arcsec"] = sep[matched].arcsec
pairs.head()

`reset_index(drop=True)` renumbers the rows from 0, so that the two tables line up row by row when we join them.

> **Watch out for duplicates.** Two Gaia sources could have the same nearest ZFOURGE object. For dense catalogues you should check for this, for example with `pairs["id"].duplicated().sum()`.

### Systematic offsets

If two catalogues have slightly different astrometry, the separations won't be centred on zero. `spherical_offsets_to()` gives the offset in RA and Dec separately, and the median tells us about any systematic shift:

In [ ]:
d_ra, d_dec = gaia_coords[matched].spherical_offsets_to(zf_coords[idx[matched]])

print(f"Median offset (ZFOURGE - Gaia): RA {np.median(d_ra.arcsec):+.3f} arcsec, Dec {np.median(d_dec.arcsec):+.3f} arcsec")

ZFOURGE positions are systematically shifted by a few tenths of an arcsecond from Gaia. This is common, because older surveys were calibrated against older reference catalogues, and Gaia's positions are for a different epoch, so the stars have moved since. A shift this small doesn't matter for a 1 arcsec match, but it would matter for, say, placing a spectrograph slit. You'd correct it by adding the median offset to the positions.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Stars or Galaxies?
  </h1>
</div>

Using `pairs`:

1. What fraction of the matched ZFOURGE objects have `star == 1`? Store it in `star_fraction`.
2. Make a mask called `moving` for matches with a significant parallax, `parallax / parallax_error > 5`. These are definitely stars in our galaxy. What fraction of *these* are flagged as stars by ZFOURGE?
3. Look at the moving objects with `star == 0`, and their `f_Ks` and `use` values. What do they have in common? *Hint: remember from Section 2 why some objects have no $K_s$ flux.*

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(star_fraction - 0.548) < 0.01, "star_fraction isn't right yet. It should be the mean of (star == 1) over all matched pairs."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
star_fraction = (pairs["star"] == 1).mean()
print(f"Fraction of all matches flagged as stars: {star_fraction:.2f}")

moving = pairs["parallax"] / pairs["parallax_error"] > 5
print(f"{moving.sum()} matches have significant parallax; "
      f"{(pairs.loc[moving, 'star'] == 1).mean():.2f} of them are flagged as stars")

print(pairs.loc[moving & (pairs["star"] == 0), ["source_id", "phot_g_mean_mag", "parallax", "id", "use", "f_Ks"]])
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

Every one of the unflagged stars has no FourStar $K_s$ flux and `use = 0`. They sit outside the area with full FourStar coverage, where the catalogue has less information to classify them. Because `use = 1` already excludes them, none of them would end up in a science sample, which is exactly what the `use` flag is for.

Cross-matching with another survey is one of the best ways to test a catalogue's flags like this. When you start on a new catalogue, it's worth checking its flags against an independent one.

<div class="practical-end" style="
  background: #f5f5f5;
  border-radius: 12px;
  padding: 20px 24px;
  text-align: center;
  border: 1px solid #ddd;
  margin: 2rem 0;
">
  <p style="font-size: 1.5rem; margin: 0 0 8px; text-align: center;">🎉</p>
  <h1 style="margin: 0 0 4px; color: #111; text-align: center;">End of Part 3</h1>
  <p style="margin: 0; color: #555; font-size: 0.9rem; text-align: center;">
    Well done! You've finished Part 3.
  </p>
</div>

<style>
@media (prefers-color-scheme: dark) {
  .practical-end { background: #1e1e1e !important; border-color: #333 !important; }
  .practical-end h1 { color: #f0f0f0 !important; }
  .practical-end p  { color: #999     !important; }
}
</style>

## What's Next?

In **Part 4** you'll learn to make publication-quality plots, including the UVJ diagram of the ZFOURGE sample you built here, colour-magnitude diagrams and images with catalogue positions overlaid.

<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/04-Plotting-for-Astronomy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Want more practice?

- [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Astropy tables](https://docs.astropy.org/en/stable/table/) and [FITS files](https://docs.astropy.org/en/stable/io/fits/)
- [astroquery documentation](https://astroquery.readthedocs.io/)
- [Learn Astropy: catalogue cross-matching tutorial](https://learn.astropy.org/tutorials/1-Coordinates-Intro.html)